<a href="https://colab.research.google.com/github/a1ephh/pytorch101/blob/main/learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset= torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader= torch.utils.data.DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader= torch.utils.data.DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

print(f"Dataset loaded! Training images: {len(train_dataset)}, Testing images: {len(test_dataset)}")


Dataset loaded! Training images: 60000, Testing images: 10000


In [2]:
class BrainNetwork(nn.Module):
  def __init__(self):
    super(BrainNetwork, self).__init__()

    self.layer1=nn.Linear(28*28, 128)

    self.relu = nn.ReLU()

    self.layer2=nn.Linear(128, 10)

  def forward(self,x):
    x=x.reshape(-1, 28*28)
    x=self.layer1(x)
    x=self.relu(x)
    x=self.layer2(x)
    return x


model = BrainNetwork()
print(model)

BrainNetwork(
  (layer1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=128, out_features=10, bias=True)
)


In [3]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=0.01)

In [4]:
epochs = 3

for epoch in range(epochs):
  running_loss=0.0

  for images, labels in train_loader:
    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")
print("Training finished!")

Epoch 1/3, Loss: 0.5792738265320182
Epoch 2/3, Loss: 0.2999451987143519
Epoch 3/3, Loss: 0.25321992913257085
Training finished!


In [5]:
weight_matrix = model.layer1.weight.data
gradient_matrix= model.layer1.weight.grad

print("--- LAYER 1 PARAMETER INSPECTION ---")
print(f"Physical shape of the weight matrix: {weight_matrix.shape}")
print(f"Physical shape of the gradient matrix: {gradient_matrix.shape}\n")

print("A tiny slice of the actual trained weights:")
print(weight_matrix[:3, :3])

# Let's check the gradients left over from the very last batch
print("\nA tiny slice of the remaining gradients in the '.grad' pocket:")
print(gradient_matrix[:3, :3])

--- LAYER 1 PARAMETER INSPECTION ---
Physical shape of the weight matrix: torch.Size([128, 784])
Physical shape of the gradient matrix: torch.Size([128, 784])

A tiny slice of the actual trained weights:
tensor([[-0.0265, -0.0125,  0.0079],
        [-0.0077, -0.0326, -0.0313],
        [-0.0283,  0.0059,  0.0239]])

A tiny slice of the remaining gradients in the '.grad' pocket:
tensor([[-0.0007, -0.0007, -0.0007],
        [ 0.0006,  0.0006,  0.0006],
        [ 0.0003,  0.0003,  0.0003]])


In [6]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy of the network on the 10,000 test images: {accuracy:.2f}%")

Accuracy of the network on the 10,000 test images: 93.41%


In [7]:
import torch

def get_split_mnist(dataset, allowed_digits, batch_size, shuffle):
    mask = torch.tensor([label in allowed_digits for _, label in dataset])
    true_indices = torch.where(mask)[0]

    filtered_subset = torch.utils.data.Subset(dataset, true_indices)
    return torch.utils.data.DataLoader(filtered_subset, batch_size=batch_size, shuffle=shuffle)

print("Blueprint function 'get_split_mnist' successfully created!")

Blueprint function 'get_split_mnist' successfully created!


In [8]:
task1_digits = [0, 1, 2, 3, 4]
task2_digits = [5, 6, 7, 8, 9]

task2_train_loader = get_split_mnist(train_dataset, task2_digits, batch_size=64, shuffle=True)
task2_test_loader = get_split_mnist(test_dataset, task2_digits, batch_size=64, shuffle=False)

task1_train_loader = get_split_mnist(train_dataset, task1_digits, batch_size=64, shuffle=True)
task1_test_loader = get_split_mnist(test_dataset, task1_digits, batch_size=64, shuffle=False)

print(f"Initialization complete!")
print(f"-> task2_train_loader is alive with {len(task2_train_loader)} batches of high digits (5-9).")
print(f"-> task1_test_loader is alive with {len(task1_test_loader)} batches of low digits (0-4).")

Initialization complete!
-> task2_train_loader is alive with 460 batches of high digits (5-9).
-> task1_test_loader is alive with 81 batches of low digits (0-4).


In [9]:
epochs_task2 = 2

print("Training exclusively on digits 5-9...")
for epoch in range(epochs_task2):
    running_loss = 0.0
    for images, labels in task2_train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Task 2 - Epoch {epoch+1}/{epochs_task2} - Loss: {running_loss/len(task2_train_loader):.4f}")

print("Training on Task 2 finished!\n")


print("Testing the model's memory on Task 1 (Digits 0-4)...")
model.eval()
correct_t1 = 0
total_t1 = 0

with torch.no_grad():
    for images, labels in task1_test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total_t1 += labels.size(0)
        correct_t1 += (predicted == labels).sum().item()

accuracy_t1 = 100 * correct_t1 / total_t1
print(f"Final Accuracy on Task 1 (Digits 0-4): {accuracy_t1:.2f}%")

Training exclusively on digits 5-9...
Task 2 - Epoch 1/2 - Loss: 0.1427
Task 2 - Epoch 2/2 - Loss: 0.1178
Training on Task 2 finished!

Testing the model's memory on Task 1 (Digits 0-4)...
Final Accuracy on Task 1 (Digits 0-4): 30.38%


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. HARD REMAKE OF THE MODEL AND OPTIMIZER
print("1. Training pristine baseline on Task 1...")
ewc_model = BrainNetwork()
ewc_optimizer = optim.SGD(ewc_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Train Task 1
ewc_model.train()
for epoch in range(3):
    for images, labels in task1_train_loader:
        ewc_optimizer.zero_grad()
        loss = criterion(ewc_model(images), labels)
        loss.backward()
        ewc_optimizer.step()

# 2. ISOLATED SNAPSHOT
task1_weight_snapshot = {name: param.data.clone().detach()
                         for name, param in ewc_model.named_parameters() if param.requires_grad}

# 3. FRESH FISHER CALCULATION (CRITICAL WIPING)
print("2. Calculating clean, normalized Fisher scores...")
fisher_matrix = {name: torch.zeros_like(param.data)
                 for name, param in ewc_model.named_parameters() if param.requires_grad}

ewc_model.eval()
# Absolute zero-grad sweep before we touch anything
ewc_optimizer.zero_grad()
for name, param in ewc_model.named_parameters():
    if param.grad is not None:
        param.grad.zero_()

for images, labels in task1_train_loader:
    ewc_model.zero_grad()
    loss = criterion(ewc_model(images), labels)
    loss.backward()
    for name, param in ewc_model.named_parameters():
        if param.requires_grad:
            fisher_matrix[name] += (param.grad.data ** 2) / len(task1_train_loader)

# Normalize
for name in fisher_matrix:
    max_val = torch.max(fisher_matrix[name])
    if max_val > 0:
        fisher_matrix[name] = fisher_matrix[name] / max_val

# 4. TRAINING TASK 2 WITH A HIGHLY STABLE EQUILIBRIUM LAMBDA
ewc_lambda = 50.0
ewc_optimizer = optim.SGD(ewc_model.parameters(), lr=0.01)

ewc_model.train()
print(f"3. Training on Task 2 with EWC Shield (Lambda = {ewc_lambda})...")
for epoch in range(2):
    running_loss = 0.0
    for images, labels in task2_train_loader:
        ewc_optimizer.zero_grad()
        base_loss = criterion(ewc_model(images), labels)

        ewc_penalty = 0.0
        for name, param in ewc_model.named_parameters():
            if param.requires_grad:
                weight_diff = param - task1_weight_snapshot[name]
                ewc_penalty += torch.sum(fisher_matrix[name] * (weight_diff ** 2))

        total_loss = base_loss + (ewc_lambda / 2.0) * ewc_penalty
        total_loss.backward()
        ewc_optimizer.step()
        running_loss += total_loss.item()
    print(f"   Epoch {epoch+1}/2 - Total Loss: {running_loss/len(task2_train_loader):.4f}")

# 5. FINAL MEMORY TEST
ewc_model.eval()
correct_ewc, total_ewc = 0, 0
with torch.no_grad():
    for images, labels in task1_test_loader:
        outputs = ewc_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total_ewc += labels.size(0)
        correct_ewc += (predicted == labels).sum().item()

print(f"\n🎯 Pure Post-Reset Accuracy on Task 1: {100 * correct_ewc / total_ewc:.2f}%")

1. Training pristine baseline on Task 1...
2. Calculating clean, normalized Fisher scores...
3. Training on Task 2 with EWC Shield (Lambda = 50.0)...
   Epoch 1/2 - Total Loss: 0.8207
   Epoch 2/2 - Total Loss: 0.3791

🎯 Pure Post-Reset Accuracy on Task 1: 9.85%
